# Лабораторная работа 4

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [41]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

%matplotlib inline
device = "/GPU:0" if tf.config.list_physical_devices("GPU") else "/CPU:0"

# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы. 

In [42]:
def load_mnist(num_training=59000, num_validation=1000, num_test=10000):
    """
    Fetch the MNIST dataset from the web and perform preprocessing to prepare
    it for the two-layer neural net classifier. These are the same steps as
    we used for the SVM, but condensed to a single function.
    """
    # Load the raw MNIST dataset and use appropriate data types and shapes
    mnist = tf.keras.datasets.mnist.load_data()
    (X_train, y_train), (X_test, y_test) = mnist
    X_train = np.asarray(X_train, dtype=np.float32)
    X_train = np.expand_dims(X_train, axis=-1)
    y_train = np.asarray(y_train, dtype=np.int32).flatten()
    X_test = np.asarray(X_test, dtype=np.float32)
    X_test = np.expand_dims(X_test, axis=-1)
    y_test = np.asarray(y_test, dtype=np.int32).flatten()

    # Subsample the data
    mask = range(num_training, num_training + num_validation)
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = range(num_training)
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = range(num_test)
    X_test = X_test[mask]
    y_test = y_test[mask]

    # Normalize the data: subtract the mean pixel and divide by std
    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)
    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_mnist()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

Train data shape:  (59000, 28, 28, 1)
Train labels shape:  (59000,) int32
Validation data shape:  (1000, 28, 28, 1)
Validation labels shape:  (1000,)
Test data shape:  (10000, 28, 28, 1)
Test labels shape:  (10000,)


In [43]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y
        
        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[i:i+B], self.y[i:i+B]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [44]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 28, 28, 1) (64,)
1 (64, 28, 28, 1) (64,)
2 (64, 28, 28, 1) (64,)
3 (64, 28, 28, 1) (64,)
4 (64, 28, 28, 1) (64,)
5 (64, 28, 28, 1) (64,)
6 (64, 28, 28, 1) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети. 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [45]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()        
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()
    
    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации. 

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU 
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU 
5. Полносвязный слой 
6. Функция активации Softmax 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [46]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        self.conv1 = tf.keras.layers.Conv2D(filters=channel_1, kernel_size=(5,5), padding='same', activation='relu')
        self.conv2 = tf.keras.layers.Conv2D(filters=channel_2, kernel_size=(3,3), padding='same', activation='relu')
        self.flatten = tf.keras.layers.Flatten()
        self.fc = tf.keras.layers.Dense(num_classes, activation='softmax')
        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        
    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.flatten(x)
        scores = self.fc(x)
        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################        
        return scores

In [47]:
def test_ThreeLayerConvNet():    
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [48]:
def train_part34(
    model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False, print_every=100
):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.

    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for

    Returns: Nothing, but prints progress during trainingn
    """
    with tf.device(device):
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

        model = model_init_fn()
        optimizer = optimizer_init_fn()

        train_loss = tf.keras.metrics.Mean(name="train_loss")
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name="train_accuracy")

        val_loss = tf.keras.metrics.Mean(name="val_loss")
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name="val_accuracy")

        t = 0
        for epoch in range(num_epochs):
            # Reset the metrics - https://www.tensorflow.org/alpha/guide/migration_guide#new-style_metrics
            train_loss.reset_state()
            train_accuracy.reset_state()

            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:
                    # Use the model function to build the forward pass.
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)

                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

                    # Update the metrics
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)

                    if t % print_every == 0:
                        val_loss.reset_state()
                        val_accuracy.reset_state()
                        for test_x, test_y in val_dset:
                            # During validation at end of epoch, training set to False
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)

                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)

                        template = "Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}"
                        print(
                            template.format(
                                t,
                                epoch + 1,
                                train_loss.result(),
                                train_accuracy.result() * 100,
                                val_loss.result(),
                                val_accuracy.result() * 100,
                            )
                        )
                    t += 1

In [49]:
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.2311348915100098, Accuracy: 9.375, Val Loss: 2.9131171703338623, Val Accuracy: 21.399999618530273
Iteration 100, Epoch 1, Loss: 0.642890453338623, Accuracy: 80.64665985107422, Val Loss: 0.3586978316307068, Val Accuracy: 90.69999694824219
Iteration 200, Epoch 1, Loss: 0.5092378258705139, Accuracy: 85.02021789550781, Val Loss: 0.2923072576522827, Val Accuracy: 92.19999694824219
Iteration 300, Epoch 1, Loss: 0.4519922435283661, Accuracy: 86.60713958740234, Val Loss: 0.27087005972862244, Val Accuracy: 92.5999984741211
Iteration 400, Epoch 1, Loss: 0.40788280963897705, Accuracy: 87.9403076171875, Val Loss: 0.22099213302135468, Val Accuracy: 94.30000305175781
Iteration 500, Epoch 1, Loss: 0.3841506540775299, Accuracy: 88.67577362060547, Val Loss: 0.2232821136713028, Val Accuracy: 94.80000305175781
Iteration 600, Epoch 1, Loss: 0.3604087829589844, Accuracy: 89.37448120117188, Val Loss: 0.22105179727077484, Val Accuracy: 94.5999984741211
Iteration 700, Epoch 1, Lo

Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 . 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [50]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10

def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    return tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9, nesterov=True)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.296621322631836, Accuracy: 10.9375, Val Loss: 2.2895195484161377, Val Accuracy: 11.5
Iteration 100, Epoch 1, Loss: 0.696721613407135, Accuracy: 80.29084014892578, Val Loss: 0.3331558108329773, Val Accuracy: 91.0999984741211
Iteration 200, Epoch 1, Loss: 0.5287120938301086, Accuracy: 85.01243591308594, Val Loss: 0.26213762164115906, Val Accuracy: 93.5
Iteration 300, Epoch 1, Loss: 0.4392487406730652, Accuracy: 87.44808959960938, Val Loss: 0.16301339864730835, Val Accuracy: 96.20000457763672
Iteration 400, Epoch 1, Loss: 0.37024539709091187, Accuracy: 89.38980865478516, Val Loss: 0.13806527853012085, Val Accuracy: 97.0999984741211
Iteration 500, Epoch 1, Loss: 0.327466756105423, Accuracy: 90.64683532714844, Val Loss: 0.13267216086387634, Val Accuracy: 96.5999984741211
Iteration 600, Epoch 1, Loss: 0.29289835691452026, Accuracy: 91.60773468017578, Val Loss: 0.12277726829051971, Val Accuracy: 97.29999542236328
Iteration 700, Epoch 1, Loss: 0.26723790168762207,

# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [51]:
learning_rate = 1e-2

def model_init_fn():
    input_shape = (28, 28, 1)
    hidden_layer_size, num_classes = 4000, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax', 
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate) 

train_part34(model_init_fn, optimizer_init_fn)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Iteration 0, Epoch 1, Loss: 3.263150930404663, Accuracy: 14.0625, Val Loss: 3.0799124240875244, Val Accuracy: 24.5
Iteration 100, Epoch 1, Loss: 0.6626821756362915, Accuracy: 79.93502807617188, Val Loss: 0.3687707483768463, Val Accuracy: 89.0
Iteration 200, Epoch 1, Loss: 0.5228331685066223, Accuracy: 84.55379486083984, Val Loss: 0.2736208438873291, Val Accuracy: 92.19999694824219
Iteration 300, Epoch 1, Loss: 0.46255484223365784, Accuracy: 86.35797119140625, Val Loss: 0.2539360225200653, Val Accuracy: 93.5
Iteration 400, Epoch 1, Loss: 0.4174218773841858, Accuracy: 87.7454833984375, Val Loss: 0.2134777009487152, Val Accuracy: 94.4000015258789
Iteration 500, Epoch 1, Loss: 0.39350980520248413, Accuracy: 88.45122528076172, Val Loss: 0.21850302815437317, Val Accuracy: 94.5999984741211
Iteration 600, Epoch 1, Loss: 0.36903098225593567, Accuracy: 89.17169189453125, Val Loss: 0.21991179883480072, Val Accuracy: 94.4000015258789
Iteration 700, Epoch 1, Loss: 0.3510836362838745, Accuracy: 89.6

Альтернативный менее гибкий способ обучения:

In [52]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

922/922 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.3223 - sparse_categorical_accuracy: 0.9042 - val_loss: 0.1788 - val_sparse_categorical_accuracy: 0.9570
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2163 - sparse_categorical_accuracy: 0.9358


[0.21633806824684143, 0.9358000159263611]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [53]:
def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    channel_1 = 32
    channel_2 = 16
    num_classes = 10
    input_shape = (28, 28, 1)

    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(channel_1, (5, 5), padding='same', activation='relu', input_shape=input_shape),
        tf.keras.layers.Conv2D(channel_2, (3, 3), padding='same', activation='relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model

learning_rate = 5e-4
def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Iteration 0, Epoch 1, Loss: 2.2837700843811035, Accuracy: 15.625, Val Loss: 2.2979583740234375, Val Accuracy: 14.100000381469727
Iteration 100, Epoch 1, Loss: 2.2531821727752686, Accuracy: 18.208539962768555, Val Loss: 2.181947708129883, Val Accuracy: 24.100000381469727
Iteration 200, Epoch 1, Loss: 2.1917378902435303, Accuracy: 26.85012435913086, Val Loss: 2.023660898208618, Val Accuracy: 48.89999771118164
Iteration 300, Epoch 1, Loss: 2.114032745361328, Accuracy: 35.77138900756836, Val Loss: 1.7600009441375732, Val Accuracy: 67.79999542236328
Iteration 400, Epoch 1, Loss: 1.9907225370407104, Accuracy: 43.68765640258789, Val Loss: 1.323927402496338, Val Accuracy: 80.0
Iteration 500, Epoch 1, Loss: 1.8425147533416748, Accuracy: 50.15593719482422, Val Loss: 0.9060112237930298, Val Accuracy: 86.29999542236328
Iteration 600, Epoch 1, Loss: 1.6866308450698853, Accuracy: 55.20486831665039, Val Loss: 0.6431777477264404, Val Accuracy: 88.20000457763672
Iteration 700, Epoch 1, Loss: 1.54973351

In [54]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

922/922 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 0.3280 - sparse_categorical_accuracy: 0.9045 - val_loss: 0.1508 - val_sparse_categorical_accuracy: 0.9730
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.1366 - sparse_categorical_accuracy: 0.9610


[0.13658146560192108, 0.9610000252723694]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры. 

Ниже представлен пример для полносвязной сети. 

In [55]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):  
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)
    
    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)
    
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_two_layer_fc_functional()

(64, 10)


In [56]:
input_shape = (28, 28, 1)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.141465902328491, Accuracy: 7.8125, Val Loss: 2.7582156658172607, Val Accuracy: 17.899999618530273
Iteration 100, Epoch 1, Loss: 0.6581016182899475, Accuracy: 79.93502807617188, Val Loss: 0.3471866250038147, Val Accuracy: 89.5
Iteration 200, Epoch 1, Loss: 0.518541157245636, Accuracy: 84.48383331298828, Val Loss: 0.2865385413169861, Val Accuracy: 91.69999694824219
Iteration 300, Epoch 1, Loss: 0.45841309428215027, Accuracy: 86.21781158447266, Val Loss: 0.25393879413604736, Val Accuracy: 93.0
Iteration 400, Epoch 1, Loss: 0.41308826208114624, Accuracy: 87.67534637451172, Val Loss: 0.21066680550575256, Val Accuracy: 95.20000457763672
Iteration 500, Epoch 1, Loss: 0.3876882493495941, Accuracy: 88.4668197631836, Val Loss: 0.2144334763288498, Val Accuracy: 94.9000015258789
Iteration 600, Epoch 1, Loss: 0.36392703652381897, Accuracy: 89.15349578857422, Val Loss: 0.2188130021095276, Val Accuracy: 94.5999984741211
Iteration 700, Epoch 1, Loss: 0.3465171754360199, A

Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут). 

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

Изначальная сеть (для сравнения)

In [57]:
channel_1 = 32
channel_2 = 16
num_classes = 10
input_shape = (28, 28, 1)

model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=128, epochs=10, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

Epoch 1/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.3891 - sparse_categorical_accuracy: 0.8860 - val_loss: 0.2071 - val_sparse_categorical_accuracy: 0.9440
Epoch 2/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.2246 - sparse_categorical_accuracy: 0.9368 - val_loss: 0.1625 - val_sparse_categorical_accuracy: 0.9610
Epoch 3/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.1854 - sparse_categorical_accuracy: 0.9483 - val_loss: 0.1377 - val_sparse_categorical_accuracy: 0.9690
Epoch 4/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.1605 - sparse_categorical_accuracy: 0.9562 - val_loss: 0.1347 - val_sparse_categorical_accuracy: 0.9690
Epoch 5/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.1426 - sparse_categorical_accuracy: 0.9611 - val_loss: 0.1208 - val_sparse_categorical_accuracy: 0.9710
Epoch 6/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.1287 - sparse_categorical_accuracy: 0.9654 - val_loss: 0.1181 - val_sparse_categorical_accuracy: 0.9710
Epoc

[0.11247597634792328, 0.9686999917030334]

### Эксперимент 1
Добавление MaxPooling2D

In [58]:
class CustomConvNet1(tf.keras.Model):
    def __init__(self) -> None:
        super(CustomConvNet1, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on mnist                      #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        self.conv1 = tf.keras.layers.Conv2D(32, (5, 5), padding='same', activation='relu')
        self.pool1 = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))

        self.conv2 = tf.keras.layers.Conv2D(16, (3, 3), padding='same', activation='relu')
        self.pool2 = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))

        self.flatten = tf.keras.layers.Flatten()
        self.fc = tf.keras.layers.Dense(10, activation='softmax')

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on mnist                  #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(input_tensor)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.pool2(x)
        x = self.flatten(x)
        x = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


model1 = CustomConvNet1()
model1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
)

history1 = model1.fit(
    X_train,
    y_train,
    batch_size=128,
    epochs=10,
    validation_data=(X_val, y_val),
    verbose=1,
)

test_loss, test_acc = model1.evaluate(X_test, y_test, verbose=0)
print(f"Model 1 - Test Accuracy: {test_acc:.4f}")

Epoch 1/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 0.2642 - sparse_categorical_accuracy: 0.9238 - val_loss: 0.1040 - val_sparse_categorical_accuracy: 0.9830
Epoch 2/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0734 - sparse_categorical_accuracy: 0.9774 - val_loss: 0.0970 - val_sparse_categorical_accuracy: 0.9800
Epoch 3/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0562 - sparse_categorical_accuracy: 0.9827 - val_loss: 0.0694 - val_sparse_categorical_accuracy: 0.9900
Epoch 4/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0464 - sparse_categorical_accuracy: 0.9851 - val_loss: 0.0759 - val_sparse_categorical_accuracy: 0.9870
Epoch 5/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0401 - sparse_categorical_accuracy: 0.9872 - val_loss: 0.0670 - val_sparse_categorical_accuracy: 0.9900
Epoch 6/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0359 - sparse_categorical_accuracy: 0.9887 - val_loss: 0.0645 - val_sparse_categorical_accuracy: 0.9890
Epoc

### Эксперимент 2
Добавление Batch Normalization и Dropout

In [59]:
class CustomConvNet2(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet2, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on mnist                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        self.conv1 = tf.keras.layers.Conv2D(32, (5, 5), padding='same')
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.relu1 = tf.keras.layers.Activation('relu')
        
        self.conv2 = tf.keras.layers.Conv2D(16, (3, 3), padding='same')
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.relu2 = tf.keras.layers.Activation('relu')
        
        self.flatten = tf.keras.layers.Flatten()
        self.dropout = tf.keras.layers.Dropout(0.5) 
        self.fc = tf.keras.layers.Dense(10, activation='softmax')
        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
    
    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on mnist                  #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(input_tensor)
        x = self.bn1(x, training=training)
        x = self.relu1(x)
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = self.relu2(x)
        x = self.flatten(x)
        x = self.dropout(x, training=training)
        x = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


model2 = CustomConvNet2()

model2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
)

history2 = model2.fit(
    X_train,
    y_train,
    batch_size=128,
    epochs=10,
    validation_data=(X_val, y_val),
    verbose=1,
)

test_loss, test_acc = model2.evaluate(X_test, y_test, verbose=0)
print(f"Model 2 - Test Accuracy: {test_acc:.4f}")

Epoch 1/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - loss: 0.1913 - sparse_categorical_accuracy: 0.9416 - val_loss: 0.1042 - val_sparse_categorical_accuracy: 0.9770
Epoch 2/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0737 - sparse_categorical_accuracy: 0.9780 - val_loss: 0.0741 - val_sparse_categorical_accuracy: 0.9880
Epoch 3/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0537 - sparse_categorical_accuracy: 0.9827 - val_loss: 0.0547 - val_sparse_categorical_accuracy: 0.9920
Epoch 4/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0448 - sparse_categorical_accuracy: 0.9861 - val_loss: 0.0667 - val_sparse_categorical_accuracy: 0.9910
Epoch 5/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.0386 - sparse_categorical_accuracy: 0.9878 - val_loss: 0.0632 - val_sparse_categorical_accuracy: 0.9890
Epoch 6/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0352 - sparse_categorical_accuracy: 0.9881 - val_loss: 0.0616 - val_sparse_categorical_accuracy: 0.9900
Ep

Опишите все эксперименты, результаты. Сделайте выводы.

В 1 эксперименте в базовую структуру сверточной сети были добавлены слои MaxPooling2D с размером окна 2x2 после каждого сверточного слоя. Операция пулинга используется для уменьшения пространственной размерности карт признаков. Это позволяет сети стать инвариантной к небольшим смещениям объектов на изображении, а также сокращает количество обучаемых параметров перед полносвязным слоем Dense.
Результаты: Уже на первой эпохе точность на тренировочной выборке составила 92.38%, а на валидационной — 98.30%.
К 10-й эпохе точность достигла 99.27% (обучение) и 99.00% (валидация).
Итоговая точность на независимой тестовой выборке составила 98.97%.

В 2 эксперименте после каждого сверточного слоя был добавлен слой BatchNormalization. Кроме того, перед финальным слоем классификации был добавлен слой Dropout, отключающий 50% нейронов.
Пакетная нормализация центрирует и масштабирует данные внутри каждого батча. Слой Dropout выступает в роли регуляризатора: случайное отключение связей заставляет сеть не заучивать тренировочные примеры, а выучивать более общие, устойчивые признаки.
Результаты: модель стартовала с очень высоких показателей - на первой эпохе точность обучения составила 94.16%, а валидации — 97.70%.
К 10-й эпохе разница между тренировочной (99.10%) и валидационной (99.30%) точностью оказалась минимальной. Итоговая точность на тестовой выборке составила 98.94%.
Также в обоих экспериментах использовался оптимизатор Adam, так как он позволяет адаптировать скорость обучения для каждого параметра, что делает процесс оптимизации более стабильным по сравнению с классическим градиентным спуском.   

Для оценки эффективности модификаций было проаедено сраавнение с классической трехслойная CNN. По результатам тестов с размером батча 128: Классическая модель достигла точности 96.87% на тестовой выборке, а экспериментальные модели показали результат около 98.9%, что значительно выше базового уровня.   

*Выводы*: Обе архитектуры справились с задачей классификации набора данных MNIST. Требуемый порог точности в 70% был значительно превышен: обе экспериментальные модели достигли результатов около 99%, продемонстрировав превосходство над классической версией сети.